# Reward-guided SELECTION survey set (variant A) — no training

For every one of the 178 prompts the shared pool holds K=16 scored samples (seeds 1000–1015).
This notebook builds the survey triplet **from the pool itself**:

| arm | image | question it answers |
|---|---|---|
| `random` | one seed drawn at random | what the personalised model gives without any guidance |
| `sel_aes` | the seed the **aesthetic** composite (NIMA + LAION-aes) ranks first | do people see what the aesthetic predictor sees? |
| `sel_tech` | the seed the **technical** composite (MUSIQ + TOPIQ-NR) ranks first | do people see what the technical predictor sees? |

Three different images per prompt (the random seed is drawn from the non-selected ones; when both axes
pick the same seed the technical arm takes its runner-up — `tech_fallback` in `selection.csv`).
Everything is CPU-only; it needs the pool **generated and scored** (`sd35_dpo_pool_extend.ipynb`, sections 0–4).

Output: `Dataset/RufGen/generated_v2_sel/{random,sel_aes,sel_tech}/SD35/{color}/{top}/seed_XXXX.png`
+ `selection.csv` + `selection_summary.json`; then the mos-eval catalogue in selection mode.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, json, glob, csv

DATASET  = '/content/drive/MyDrive/Doktorat/Dokumetacja/reward-GENERATED/Dataset/RufGen'
POOL     = f'{DATASET}/pool'
MANIFEST = f'{DATASET}/tables/guitar_manifest.csv'
SEL      = f'{DATASET}/generated_v2_sel'
MOS      = '/content/drive/MyDrive/Doktorat/Dokumetacja/reward-GENERATED/mos-eval'
REPO_SRC = '/content/drive/MyDrive/Doktorat/Dokumetacja/reward-GENERATED/code/sd35-dpo'
REPO = '/content/sd35-dpo'
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(REPO_SRC, REPO, ignore=shutil.ignore_patterns('.git', '__pycache__', '.DS_Store'))
%cd {REPO}
!pip install -q pyyaml torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip install -q pyyaml
for name, path in [('pool meta', f'{POOL}/meta.json'), ('pool scores', f'{POOL}/scores.csv'), ('manifest', MANIFEST)]:
    print(('OK      ' if os.path.exists(path) else 'MISSING ') + f'{name}: {path}')

## 1. Is the pool complete and scored?

In [ ]:
import pandas as pd
meta = json.load(open(f'{POOL}/meta.json')); sc = pd.read_csv(f'{POOL}/scores.csv')
print(meta)
print(len(sc), 'scored rows;', sc.groupby('prompt_idx').size().value_counts().to_dict(), '(K per prompt: count)')
need = ['nima', 'laion_aes', 'musiq', 'topiq_nr']
print('metrics present:', [m for m in need if m in sc.columns], '| missing:', [m for m in need if m not in sc.columns])
print('images on disk:', len(glob.glob(f'{POOL}/images/*.png')))

## 2. Build the selection set (CPU, seconds)

In [ ]:
!python scripts/build_selection.py --pool {POOL} --manifest {MANIFEST} --out {SEL} --rng 5 \
    --aes-config configs/aesthetic.yaml --tech-config configs/technical.yaml

In [ ]:
s = json.load(open(f'{SEL}/selection_summary.json'))
print({k: s[k] for k in ['n_prompts', 'n_images', 'axes_agree_on_best', 'axes_agree_frac', 'gap_aes_mean', 'gap_tech_mean',
                         'random_rank_aes_mean', 'random_rank_tech_mean', 'n_copied']})
sel = pd.read_csv(f'{SEL}/selection.csv')
print('how often each pool seed wins the aesthetic arm:', sel.sel_aes_seed.value_counts().head(5).to_dict())
sel[['color', 'top', 'random_seed', 'sel_aes_seed', 'sel_tech_seed', 'random_aes', 'sel_aes_aes', 'random_tech', 'sel_tech_tech', 'tech_fallback']].head(8)

### Quick look: random / sel_aes / sel_tech

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
rows = sel.sample(4, random_state=0)
fig, axs = plt.subplots(len(rows), 3, figsize=(12, 4 * len(rows)))
for i, (_, r) in enumerate(rows.iterrows()):
    for j, arm in enumerate(['random', 'sel_aes', 'sel_tech']):
        p = f"{SEL}/{arm}/SD35/{r.color}/{r.top}/seed_{int(r[f'{arm}_seed']):04d}.png"
        if os.path.exists(p): axs[i, j].imshow(Image.open(p))
        axs[i, j].set_title(f"{r.color}/{r.top} {arm} (aes {r[f'{arm}_aes']:+.2f}, tech {r[f'{arm}_tech']:+.2f})", fontsize=7)
        axs[i, j].axis('off')
plt.tight_layout(); plt.show()

## 3. Catalogue for the survey platform (selection mode)

Writes `mos-eval/data/tasks_catalog_sel.json` (178 tasks, different seed per arm, `meta.selection=true`)
and samples the 10-task survey into `mos-eval/data/tasks.json`. Redeploy mos-eval afterwards.

In [ ]:
%cd {MOS}
!python scripts/build_catalog.py --selection {SEL}/selection.csv \
    --base https://storage.googleapis.com/rufai-471714-images/rufgen/generated_v2_sel --out data/tasks_catalog_sel.json
!python scripts/sample_survey.py --catalog data/tasks_catalog_sel.json --rng 5
%cd {REPO}

## 4. Upload the 534 images to the bucket

In [ ]:
# bucket in the COMPANY project (deploy/BUCKET-FIRMOWY.md); authenticate with the company account
BUCKET = 'rufai-471714-images'
# from google.colab import auth; auth.authenticate_user()
# !gcloud config set project rufai-471714
# !gcloud storage rsync -r -x '.*\.(csv|json)$' {SEL} gs://{BUCKET}/rufgen/generated_v2_sel
# !curl -sI https://storage.googleapis.com/{BUCKET}/rufgen/generated_v2_sel/random/SD35/$(ls {SEL}/random/SD35 | head -1)/plain/seed_1005.png | head -1